# Appendix Export Notebook

This notebook produces thesis-ready appendix assets automatically from the current result files.

It does three things:

1. loads the current benchmark and prompt-ranking outputs,
2. creates rounded appendix tables,
3. copies the selected figures into a single export folder for easy thesis use.

Run the notebook from top to bottom whenever the result CSV files change.


In [ ]:
from pathlib import Path
from shutil import copy2
import pandas as pd
from IPython.display import display

BASE = Path('..').resolve()
OUTPUTS = BASE / 'outputs'
EXPORT_DIR = OUTPUTS / 'appendix_export'
FIG_EXPORT_DIR = EXPORT_DIR / 'figures'
TABLE_EXPORT_DIR = EXPORT_DIR / 'tables'
FIG_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

## 1. Load Current Result Files

In [ ]:
text_summary = pd.read_csv(OUTPUTS / 'human_baseline_summaries' / 'text_metric_summary.csv')
pairwise_summary = pd.read_csv(OUTPUTS / 'human_baseline_summaries' / 'pairwise_summary.csv')
pilot_ranked = pd.read_csv(OUTPUTS / 'pilot_ranking_local' / 'pilot_condition_ranked.csv')

display(text_summary.head())
display(pairwise_summary.head())
display(pilot_ranked.head())

## 2. Build Rounded Appendix Tables

These are the compact versions most suitable for the thesis appendix. They are rounded to two decimals for readability.


In [ ]:
human_level = text_summary[text_summary['level'].isin(['Advanced','Elementary'])][['level','aoa_mean','concreteness_mean','imageability_mean','fk_grade_mean','mtld_mean','cli_mean','tier2_proxy_token_ratio_mean']].copy()
human_level.columns = ['Level','AoA','Concreteness','Imageability','Flesch-Kincaid','MTLD','CLI','Tier2Proxy']
human_level = human_level.round(2)

human_pairwise = pairwise_summary[['semantic_similarity_sbert_mean','delta_cli_mean','delta_aoa_mean','delta_concreteness_mean','delta_imageability_mean','delta_fk_grade_mean','delta_mtld_mean','delta_tier2_proxy_token_ratio_mean']].copy()
human_pairwise.columns = ['SBERT','DeltaCLI','DeltaAoA','DeltaConcreteness','DeltaImageability','DeltaFKGL','DeltaMTLD','DeltaTier2Proxy']
human_pairwise = human_pairwise.round(2)

prompt_top5 = pilot_ranked[['strategy','temperature','json_success_rate','stability_score','semantic_similarity_sbert','human_benchmark_score','overall_score']].head(5).copy()
prompt_top5.columns = ['Strategy','Temperature','JSONSuccess','StabilityScore','SBERT','BenchmarkScore','OverallScore']
prompt_top5 = prompt_top5.round(2)

display(human_level)
display(human_pairwise)
display(prompt_top5)

## 3. Save Appendix Tables

This cell writes the rounded appendix tables to `outputs/appendix_export/tables`.


In [ ]:
human_level_path = TABLE_EXPORT_DIR / 'appendix_human_level_summary.csv'
human_pairwise_path = TABLE_EXPORT_DIR / 'appendix_human_pairwise_summary.csv'
prompt_top5_path = TABLE_EXPORT_DIR / 'appendix_prompt_top5.csv'

human_level.to_csv(human_level_path, index=False)
human_pairwise.to_csv(human_pairwise_path, index=False)
prompt_top5.to_csv(prompt_top5_path, index=False)

print(human_level_path)
print(human_pairwise_path)
print(prompt_top5_path)

## 4. Export Figures

This cell copies the selected figure files into `outputs/appendix_export/figures` so they are grouped in one clean folder.


In [ ]:
figure_paths = [
    OUTPUTS / 'presentation_figures' / 'human_benchmark_profile.png',
    OUTPUTS / 'appendix_figures' / 'prompt_score_heatmap.png',
    OUTPUTS / 'presentation_figures' / 'top5_prompt_conditions.png',
    OUTPUTS / 'presentation_figures' / 'prompt_component_breakdown.png',
]
for src in figure_paths:
    dst = FIG_EXPORT_DIR / src.name
    copy2(src, dst)
    print(dst)

## 5. Appendix Export Manifest

This final cell writes a small manifest file so you can see exactly what was exported.


In [ ]:
manifest = pd.DataFrame({
    'type': ['table','table','table','figure','figure','figure','figure'],
    'name': [
        'appendix_human_level_summary.csv',
        'appendix_human_pairwise_summary.csv',
        'appendix_prompt_top5.csv',
        'human_benchmark_profile.png',
        'prompt_score_heatmap.png',
        'top5_prompt_conditions.png',
        'prompt_component_breakdown.png',
    ]
})
manifest_path = EXPORT_DIR / 'appendix_manifest.csv'
manifest.to_csv(manifest_path, index=False)
display(manifest)
print(manifest_path)